### **Installations and Imports**

In [1]:
import importlib.util
import subprocess
import sys

def is_installed(import_name):
    return importlib.util.find_spec(import_name) is not None

# package_name_on_pip : import_name_in_python
required = {
    "unsloth": "unsloth",
    "unsloth_zoo": "unsloth_zoo",
    "trl": "trl",
    "bitsandbytes": "bitsandbytes",
    "sacrebleu": "sacrebleu",
    "evaluate": "evaluate",
}

missing = [
    pip_name
    for pip_name, import_name in required.items()
    if not is_installed(import_name)
]

print("Missing packages:", missing)

if missing:
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing,
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("All required extra packages are already installed.")

print("Minimal installation finished.")

Missing packages: ['unsloth', 'unsloth_zoo', 'trl', 'bitsandbytes', 'sacrebleu', 'evaluate']
Running: /usr/bin/python3 -m pip install -q --no-cache-dir unsloth unsloth_zoo trl bitsandbytes sacrebleu evaluate
Minimal installation finished.


In [2]:
# ============================================================
# Cell 1B — Verify environment
# ============================================================

import torch
import datasets
import transformers
import peft
import accelerate
import trl
import unsloth
import bitsandbytes
import sacrebleu

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available.")
    print("Do not start Qwen/Unsloth fine-tuning on CPU.")
    print("Go to Runtime → Change runtime type → GPU.")

print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("trl:", trl.__version__)
print("sacrebleu:", sacrebleu.__version__)

print("Environment check finished.")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
datasets: 4.3.0
transformers: 5.5.0
peft: 0.19.1
accelerate: 1.13.0
trl: 0.24.0
sacrebleu: 2.6.0
Environment check finished.


### **Paths and Configurations**

In [3]:
# ============================================================
# Cell 2 — Mount Google Drive and define paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/alexandria_qwen35_sft")
DATA_DIR    = PROJECT_DIR / "prepared_data"
RUNS_DIR    = PROJECT_DIR / "runs"
ADAPTER_DIR = PROJECT_DIR / "final_adapters"
PRED_DIR    = PROJECT_DIR / "predictions"

for p in [PROJECT_DIR, DATA_DIR, RUNS_DIR, ADAPTER_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("PRED_DIR:", PRED_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft
DATA_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data
RUNS_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs
ADAPTER_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters
PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/predictions


In [4]:
# ============================================================
# Cell 3 — Experiment configuration
# Safe rerun Experiment 2 v2: FNN/MLP group, r=16, 10 epochs, keep checkpoints
# ============================================================

import torch
import random
import numpy as np

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

MODEL_NAME = "unsloth/Qwen3.5-2B-Base"

# ------------------------------------------------------------
# Country/config selection
# ------------------------------------------------------------

SELECTED_CONFIGS_MODE = "EG_ONLY"   # "EG_ONLY" or "ALL" or "MANUAL"
MANUAL_CONFIGS = ["EG"]

# ------------------------------------------------------------
# Translation setup
# ------------------------------------------------------------

MAX_CONTEXT_TURNS = 3
USE_PREVIOUS_ENGLISH_CONTEXT = True
USE_METADATA = True

# ------------------------------------------------------------
# LoRA mode
# ------------------------------------------------------------
# "attn" = q_proj, k_proj, v_proj, o_proj
# "mlp"  = FNN / feed-forward group: gate_proj, up_proj, down_proj
# "all"  = attention + FNN / MLP

LORA_MODE = "mlp"

# ------------------------------------------------------------
# LoRA rank settings
# ------------------------------------------------------------

LORA_R = 16
LORA_ALPHA = 32

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

NUM_EPOCHS = 10
MAX_SEQ_LENGTH = 768

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03

SAVE_STEPS = 200
EVAL_STEPS = 200
LOGGING_STEPS = 10

PACKING = False

# Keep enough checkpoints so checkpoint-1000 is not deleted again.
SAVE_TOTAL_LIMIT = 20

# ------------------------------------------------------------
# Experiment naming — safe new path
# ------------------------------------------------------------

if LORA_MODE == "attn":
    LORA_GROUP_NAME = "attn_group"
elif LORA_MODE == "mlp":
    LORA_GROUP_NAME = "fnn_group"
elif LORA_MODE == "all":
    LORA_GROUP_NAME = "all_group"
else:
    raise ValueError("LORA_MODE must be 'attn', 'mlp', or 'all'.")

EXPERIMENT_NAME = (
    f"qwen35_2b_alexandria_{SELECTED_CONFIGS_MODE.lower()}_"
    f"context{MAX_CONTEXT_TURNS}_{LORA_GROUP_NAME}_r{LORA_R}_v2_10epochs"
)

OUTPUT_DIR = RUNS_DIR / EXPERIMENT_NAME
FINAL_ADAPTER_PATH = ADAPTER_DIR / EXPERIMENT_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER_PATH.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("LoRA mode:", LORA_MODE)
print("LoRA r:", LORA_R)
print("LoRA alpha:", LORA_ALPHA)
print("Save total limit:", SAVE_TOTAL_LIMIT)
print("Output dir:", OUTPUT_DIR)
print("Final adapter:", FINAL_ADAPTER_PATH)

Model: unsloth/Qwen3.5-2B-Base
Experiment: qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs
LoRA mode: mlp
LoRA r: 16
LoRA alpha: 32
Save total limit: 20
Output dir: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs
Final adapter: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs


### **Dataset Preparation**

In [5]:
# ============================================================
# Cell 4 — List Alexandria configs and load selected configs
# ============================================================

from datasets import load_dataset, get_dataset_config_names
import pandas as pd

DATASET_NAME = "UBC-NLP/alexandria"

available_configs = get_dataset_config_names(DATASET_NAME)
print("Available Alexandria configs:")
print(available_configs)

if SELECTED_CONFIGS_MODE == "EG_ONLY":
    selected_configs = ["EG"] if "EG" in available_configs else [available_configs[0]]

elif SELECTED_CONFIGS_MODE == "ALL":
    selected_configs = available_configs

elif SELECTED_CONFIGS_MODE == "MANUAL":
    selected_configs = MANUAL_CONFIGS
    missing = [c for c in selected_configs if c not in available_configs]
    if missing:
        raise ValueError(f"These configs are not available: {missing}")

else:
    raise ValueError("SELECTED_CONFIGS_MODE must be EG_ONLY, ALL, or MANUAL.")

print("\nSelected configs:")
print(selected_configs)

loaded = {}

for cfg in selected_configs:
    print(f"\nLoading config: {cfg}")
    ds_train = load_dataset(DATASET_NAME, name=cfg, split="train")
    ds_test  = load_dataset(DATASET_NAME, name=cfg, split="test")

    loaded[cfg] = {
        "train": ds_train,
        "test": ds_test,
    }

    print("Train:", ds_train)
    print("Test:", ds_test)
    print("Example keys:", ds_train[0].keys())

README.md: 0.00B [00:00, ?B/s]

Available Alexandria configs:
['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']

Selected configs:
['EG']

Loading config: EG


EG/train-00000-of-00001.parquet:   0%|          | 0.00/496k [00:00<?, ?B/s]

EG/test-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/982 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/366 [00:00<?, ? examples/s]

Train: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 982
})
Test: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 366
})
Example keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])


In [6]:
# ============================================================
# Cell 5 — Inspect one raw example
# ============================================================

sample_cfg = selected_configs[0]
sample_row = loaded[sample_cfg]["train"][0]

print("Config:", sample_cfg)
print("Keys:", sample_row.keys())

print("\nEnglish conversation:")
print(sample_row["english_conversation"])

print("\nDialectal conversation:")
print(sample_row["dialectal_conversation"])

print("\nFull row:")
sample_row

Config: EG
Keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])

English conversation:
[{'direction': 'male -> female', 'speaker': 'Wholesale Buyer', 'text': "Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?", 'turn_order': 1}, {'direction': 'female -> male', 'speaker': 'Wholesale Seller', 'text': "Good morning to you. You heard correctly. My artichokes are the best you'll find. They are top-grade, perfect for export. Let me show you a sample.", 'turn_order': 2}, {'direction': 'male -> female', 'speaker': 'Wholesale Buyer', 'text': 'Excellent. Yes, please show me. I need them to be a specific size and completely free of blemishes.', 'turn_order': 3}, {'direction': 'female -> male', 'speaker': 'Wholesale Seller', 'text': "Don't you worry. You will be very satisfied. My reputatio

{'conv_id': 'B7-1-0-120',
 'country': 'EG',
 'domain': 'Agriculture and farming',
 'dialect': 'Egyptian Arabic (Cairene) Dialect',
 'participants': 'Wholesale Buyer, Wholesale Seller',
 'english_conversation': [{'direction': 'male -> female',
   'speaker': 'Wholesale Buyer',
   'text': "Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?",
   'turn_order': 1},
  {'direction': 'female -> male',
   'speaker': 'Wholesale Seller',
   'text': "Good morning to you. You heard correctly. My artichokes are the best you'll find. They are top-grade, perfect for export. Let me show you a sample.",
   'turn_order': 2},
  {'direction': 'male -> female',
   'speaker': 'Wholesale Buyer',
   'text': 'Excellent. Yes, please show me. I need them to be a specific size and completely free of blemishes.',
   'turn_order': 3},
  {'direction': 'female -> male',
   'speaker': 'Wholesale Seller',
   'text': "D

#### Helper functions for robust extraction

In [7]:
# ============================================================
# Cell 6 — Helper functions for robust extraction
# ============================================================

def safe_get(row, keys, default=""):
    for k in keys:
        if isinstance(row, dict) and k in row and row[k] is not None:
            return row[k]
    return default

def turn_text(turn):
    if isinstance(turn, dict):
        for k in ["text", "sentence", "utterance", "content", "value"]:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
        return str(turn).strip()
    return str(turn).strip()

def turn_field(turn, keys, default=""):
    if isinstance(turn, dict):
        for k in keys:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
    return default

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return list(x)

def truncate_text(text, max_chars=1200):
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."

#### Flatten Alexandria conversations into SFT examples and Save

In [8]:
# ============================================================
# Cell 7 — Flatten Alexandria conversations
# ============================================================

def flatten_alexandria_split(ds, cfg_name, split_name, max_context_turns=3):
    records = []

    for conv_idx, row in enumerate(ds):
        english_conv = normalize_list(row["english_conversation"])
        dialect_conv = normalize_list(row["dialectal_conversation"])

        n = min(len(english_conv), len(dialect_conv))

        country = safe_get(row, ["country", "country_code"], cfg_name)
        dialect = safe_get(row, ["dialect", "dialect_label", "subdialect", "city", "variety"], "")
        domain = safe_get(row, ["domain", "topic"], "")
        persona = safe_get(row, ["persona", "roles", "speaker_roles"], "")
        conv_id = safe_get(row, ["conversation_id", "id", "dialogue_id"], f"{cfg_name}_{split_name}_{conv_idx}")

        for i in range(n):
            en_turn = english_conv[i]
            ar_turn = dialect_conv[i]

            source_text = turn_text(en_turn)
            target_text = turn_text(ar_turn)

            if not source_text or not target_text:
                continue

            prev_start = max(0, i - max_context_turns)
            prev_en_turns = english_conv[prev_start:i]

            previous_context = []
            for j, t in enumerate(prev_en_turns, start=prev_start):
                previous_context.append({
                    "turn_id": j,
                    "speaker": turn_field(t, ["speaker", "role", "speaker_role"], ""),
                    "direction": turn_field(t, ["direction", "gender_direction", "speaker_addressee_gender"], ""),
                    "text": turn_text(t),
                })

            records.append({
                "source_id": f"{cfg_name}_{split_name}_{conv_id}_{i}",
                "config": cfg_name,
                "split": split_name,
                "conversation_id": conv_id,
                "turn_id": i,

                "country": country,
                "dialect": dialect,
                "domain": domain,
                "persona": persona,

                "speaker": turn_field(en_turn, ["speaker", "role", "speaker_role"], ""),
                "gender_direction": turn_field(en_turn, ["direction", "gender_direction", "speaker_addressee_gender"], ""),

                "previous_english_turns": previous_context,
                "source_text": source_text,
                "target_arabic": target_text,
            })

    return records

train_records = []
eval_records = []

for cfg in selected_configs:
    train_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["train"],
            cfg_name=cfg,
            split_name="train",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

    eval_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["test"],
            cfg_name=cfg,
            split_name="test",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

train_df = pd.DataFrame(train_records)
eval_df = pd.DataFrame(eval_records)

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)

print("\nTrain configs:")
print(train_df["config"].value_counts())

print("\nEval configs:")
print(eval_df["config"].value_counts())

display(train_df.head())

Train shape: (3108, 14)
Eval shape: (1118, 14)

Train configs:
config
EG    3108
Name: count, dtype: int64

Eval configs:
config
EG    1118
Name: count, dtype: int64


,source_id,config,split,conversation_id,turn_id,country,dialect,domain,persona,speaker,gender_direction,previous_english_turns,source_text,target_arabic
0,EG_train_EG_train_0_0,EG,train,EG_train_0,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,[],Good morning. I'm looking to source 10 tons of...,صباح الخير، عايز عشرة طن من الخرشوف الكويس للت...
1,EG_train_EG_train_0_1,EG,train,EG_train_0,1,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Good morning to you. You heard correctly. My a...,صباح النور،سمعك مظبوط،الخرشوف بتاعي من أحسن ال...
2,EG_train_EG_train_0_2,EG,train,EG_train_0,2,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...","Excellent. Yes, please show me. I need them to...",ممتاز، لو سمحتي وريني، عايزه بمقاس واحد ومافيه...
3,EG_train_EG_train_0_3,EG,train,EG_train_0,3,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Don't you worry. You will be very satisfied. M...,متخافش، هتنبسط جدا، سمعتي جاية من الحاجة الكويسة.
4,EG_train_EG_train_1_0,EG,train,EG_train_1,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Farmer,female -> female,[],I usually use the regular granular fertilizer....,أنا عادة بستخدم السماد العادي الحبيبات. ايه فا...


In [9]:
# ============================================================
# Cell 8 — Save prepared flattened data
# ============================================================

train_jsonl = DATA_DIR / f"alexandria_train_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"
eval_jsonl  = DATA_DIR / f"alexandria_eval_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"

train_df.to_json(train_jsonl, orient="records", lines=True, force_ascii=False)
eval_df.to_json(eval_jsonl, orient="records", lines=True, force_ascii=False)

print("Saved train:", train_jsonl)
print("Saved eval:", eval_jsonl)

Saved train: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_train_eg_only_context3.jsonl
Saved eval: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_eval_eg_only_context3.jsonl


### **Build prompt and chat messages**

In [10]:
# ============================================================
# Cell 9 — Build prompt and chat messages
# ============================================================

from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Return only the translation, without explanation."
)

def build_context(previous_turns):
    if not USE_PREVIOUS_ENGLISH_CONTEXT or not previous_turns:
        return "No previous context."

    lines = []
    for i, t in enumerate(previous_turns, start=1):
        speaker = t.get("speaker", "")
        text = t.get("text", "")
        if speaker:
            lines.append(f"{i}. {speaker}: {text}")
        else:
            lines.append(f"{i}. {text}")

    return "\n".join(lines)

def build_metadata_block(row):
    if not USE_METADATA:
        return "No metadata."

    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
        ("Persona/Roles", row.get("persona", "")),
        ("Current speaker", row.get("speaker", "")),
        ("Speaker-to-addressee gender direction", row.get("gender_direction", "")),
    ]

    lines = []
    for k, v in fields:
        v = str(v).strip()
        if v:
            lines.append(f"{k}: {v}")

    return "\n".join(lines) if lines else "No metadata."

def make_user_prompt(row):
    context = build_context(row["previous_english_turns"])
    metadata = build_metadata_block(row)

    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use natural local dialectal Arabic, not Modern Standard Arabic unless it is natural in context.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

def row_to_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": row["target_arabic"]},
    ]

train_df["messages"] = train_df.apply(row_to_messages, axis=1)
eval_df["messages"]  = eval_df.apply(row_to_messages, axis=1)

train_dataset = Dataset.from_pandas(
    train_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

eval_dataset = Dataset.from_pandas(
    eval_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

print(train_dataset)
print(eval_dataset)

print("\nExample messages:")
train_dataset[0]["messages"]

Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 3108
})
Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 1118
})

Example messages:


[{'role': 'system',
  'content': 'You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Return only the translation, without explanation.'},
 {'role': 'user',
  'content': "Task:\nTranslate the current English dialogue turn into the target dialectal Arabic variety.\n\nMetadata:\nCountry/config: EG\nTarget dialect: Egyptian Arabic (Cairene) Dialect\nDomain: Agriculture and farming\nCurrent speaker: Wholesale Buyer\nSpeaker-to-addressee gender direction: male -> female\n\nPrevious English dialogue context:\nNo previous context.\n\nCurrent English turn:\nGood morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?\n\nRules:\n- Preserve the meaning exactly.\n- Use natural local dialectal Arabic, not Modern Standard Arabic unless it is natural in context.\n- Preserve names, numbers, named entities, and technical terms when appr

### **Load Qwen3.5-2B with Unsloth**

In [11]:
# ============================================================
# UPDATED Cell 10 — Load Qwen3.5-2B with Unsloth
# Prefer FastLanguageModel for text-only SFT
# ============================================================

import torch

try:
    from unsloth import FastLanguageModel
    UnslothModel = FastLanguageModel
    print("Using unsloth.FastLanguageModel")
except Exception:
    from unsloth import FastModel
    UnslothModel = FastModel
    print("Using unsloth.FastModel fallback")

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

model, tokenizer = UnslothModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = dtype,
    load_in_4bit = False,
)

# If Unsloth returns a processor-like object, extract the real tokenizer if available.
if hasattr(tokenizer, "tokenizer"):
    print("Tokenizer object has internal tokenizer. Using tokenizer.tokenizer.")
    tokenizer = tokenizer.tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded:", MODEL_NAME)
print("dtype:", dtype)
print("Tokenizer type:", type(tokenizer))
print("pad token:", tokenizer.pad_token)
print("eos token:", tokenizer.eos_token)

Using unsloth.FastLanguageModel
==((====))==  Unsloth 2026.5.7: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/4.43G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/146 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

Tokenizer object has internal tokenizer. Using tokenizer.tokenizer.
Loaded: unsloth/Qwen3.5-2B-Base
dtype: torch.float16
Tokenizer type: <class 'transformers.tokenization_utils_tokenizers.TokenizersBackend'>
pad token: <|vision_pad|>
eos token: <|endoftext|>


### **Apply Chat Template**

In [12]:
# ============================================================
# UPDATED Cell 11 — Manual SFT template
# Do NOT use tokenizer.apply_chat_template
# ============================================================

SYSTEM_MARKER = "### System:"
INSTRUCTION_MARKER = "### Instruction:"
RESPONSE_MARKER = "### Arabic translation:"

def get_message_content(messages, role):
    for m in messages:
        if m.get("role") == role:
            return m.get("content", "")
    return ""

def format_sft_text(system_text, user_text, assistant_text=None, add_eos=True):
    """
    Manual decoder-only SFT format.

    During training:
        prompt + assistant answer + EOS

    During inference:
        prompt only, ending at RESPONSE_MARKER
    """

    text = (
        f"{SYSTEM_MARKER}\n"
        f"{system_text.strip()}\n\n"
        f"{INSTRUCTION_MARKER}\n"
        f"{user_text.strip()}\n\n"
        f"{RESPONSE_MARKER}\n"
    )

    if assistant_text is not None:
        text += assistant_text.strip()

        if add_eos and tokenizer.eos_token is not None:
            text += tokenizer.eos_token

    return text

def apply_manual_sft_template(example):
    messages = example["messages"]

    system_text = get_message_content(messages, "system")
    user_text = get_message_content(messages, "user")
    assistant_text = get_message_content(messages, "assistant")

    text = format_sft_text(
        system_text=system_text,
        user_text=user_text,
        assistant_text=assistant_text,
        add_eos=True,
    )

    return {"text": text}

remove_cols_train = [c for c in train_dataset.column_names if c == "messages"]
remove_cols_eval  = [c for c in eval_dataset.column_names if c == "messages"]

train_dataset_text = train_dataset.map(
    apply_manual_sft_template,
    remove_columns=remove_cols_train,
)

eval_dataset_text = eval_dataset.map(
    apply_manual_sft_template,
    remove_columns=remove_cols_eval,
)

print(train_dataset_text)
print(eval_dataset_text)

print("\nFormatted example:")
print(train_dataset_text[0]["text"])

Map:   0%|          | 0/3108 [00:00<?, ? examples/s]

Map:   0%|          | 0/1118 [00:00<?, ? examples/s]

Dataset({
    features: ['source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic', 'text'],
    num_rows: 3108
})
Dataset({
    features: ['source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic', 'text'],
    num_rows: 1118
})

Formatted example:
### System:
You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Return only the translation, without explanation.

### Instruction:
Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Metadata:
Country/config: EG
Target dialect: Egyptian Arabic (Cairene) Dialect
Domain: Agriculture and farming
Current speaker: Wholesale Buyer
Speaker-to-addressee gender direction: male -> female

Previous English dialogue context:
No previous context.

Current English turn:
Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section,

#### **Configure LoRA target modules**

In [13]:
# ============================================================
# Cell 12 — Configure LoRA
# Safe rerun: FNN/MLP group, r=16
# ============================================================

if LORA_MODE == "attn":
    TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

elif LORA_MODE == "mlp":
    TARGET_MODULES = ["gate_proj", "up_proj", "down_proj"]

elif LORA_MODE == "all":
    TARGET_MODULES = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]

else:
    raise ValueError("LORA_MODE must be 'attn', 'mlp', or 'all'.")

print("Target modules:", TARGET_MODULES)
print("LoRA r:", LORA_R)
print("LoRA alpha:", LORA_ALPHA)

model = UnslothModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    max_seq_length = MAX_SEQ_LENGTH,
)

model.print_trainable_parameters()

Target modules: ['gate_proj', 'up_proj', 'down_proj']
LoRA r: 16
LoRA alpha: 32


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


trainable params: 9,437,184 || all params: 2,222,678,848 || trainable%: 0.4246


#### Check for existing checkpoints

In [14]:
# ============================================================
# Cell 13 — Check for existing checkpoints
# ============================================================

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = None

if OUTPUT_DIR.exists():
    last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))

if last_checkpoint:
    print("Found checkpoint:")
    print(last_checkpoint)
else:
    print("No checkpoint found. Training will start from scratch.")

Found checkpoint:
/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs/checkpoint-1600


### **Build SFTTrainer**

In [15]:
# ============================================================
# Cell 14 — Build SFTTrainer
# ============================================================

from trl import SFTTrainer, SFTConfig

sft_args = SFTConfig(
    output_dir = str(OUTPUT_DIR),

    num_train_epochs = NUM_EPOCHS,
    per_device_train_batch_size = PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size = 1,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,

    learning_rate = LEARNING_RATE,
    warmup_ratio = WARMUP_RATIO,
    lr_scheduler_type = "cosine",

    optim = "adamw_8bit",
    weight_decay = 0.01,

    logging_steps = LOGGING_STEPS,

    eval_strategy = "steps",
    eval_steps = EVAL_STEPS,

    save_strategy = "steps",
    save_steps = SAVE_STEPS,
    save_total_limit = SAVE_TOTAL_LIMIT,

    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),

    seed = SEED,
    dataset_num_proc = 2,
    report_to = "none",

    packing = PACKING,
    dataset_text_field = "text",
    max_length = MAX_SEQ_LENGTH,
)

try:
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset_text,
        eval_dataset = eval_dataset_text,
        args = sft_args,
    )
except TypeError:
    trainer = SFTTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = train_dataset_text,
        eval_dataset = eval_dataset_text,
        args = sft_args,
    )

print("Trainer ready.")
print("Output dir:", OUTPUT_DIR)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3108 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1118 [00:00<?, ? examples/s]

Trainer ready.
Output dir: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs


#### Response-only training with manual markers

In [16]:
# ============================================================
# UPDATED Cell 15 — Train on assistant response only
# Uses manual markers from Cell 11
# ============================================================

try:
    from unsloth.chat_templates import train_on_responses_only

    trainer = train_on_responses_only(
        trainer,
        instruction_part = INSTRUCTION_MARKER,
        response_part = RESPONSE_MARKER,
    )

    print("Enabled response-only training.")
    print("Instruction marker:", INSTRUCTION_MARKER)
    print("Response marker:", RESPONSE_MARKER)

except Exception as e:
    print("Could not enable response-only training.")
    print("Continuing with normal SFT.")
    print("Reason:", repr(e))

Map (num_proc=6):   0%|          | 0/3108 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/3108 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/1118 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/1118 [00:00<?, ? examples/s]

Enabled response-only training.
Instruction marker: ### Instruction:
Response marker: ### Arabic translation:


### **Training**

In [17]:
# ============================================================
# Cell 16 — Train or resume
# ============================================================

if last_checkpoint:
    print("Resuming from:", last_checkpoint)
    trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting from scratch.")
    trainer_stats = trainer.train()

print("Training finished.")
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248044}.


Resuming from: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs/checkpoint-1600


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,108 | Num Epochs = 10 | Total steps = 3,890
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 9,437,184 of 2,222,678,848 (0.42% trained)


Step,Training Loss,Validation Loss
1800,0.299322,2.357483


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs/checkpoint-1800/tokenizer_config.json.


Step,Training Loss,Validation Loss
1800,0.299322,2.357483
2000,0.141033,2.451434


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs/checkpoint-2000/tokenizer_config.json.


KeyboardInterrupt: 

### **Save final LoRA adapter**

In [18]:
# ============================================================
# Cell 17 — Find, load, and save BEST checkpoint after training
# ============================================================

from pathlib import Path
import json
import torch

def find_best_checkpoint_from_logs(output_dir):
    output_dir = Path(output_dir)

    state_files = list(output_dir.rglob("trainer_state.json"))
    if not state_files:
        raise FileNotFoundError(f"No trainer_state.json found under: {output_dir}")

    # Use the trainer_state with the largest global step
    best_state_file = None
    best_state = None
    max_global_step = -1

    for sf in state_files:
        try:
            state = json.loads(sf.read_text())
            global_step = int(state.get("global_step", -1))

            if global_step > max_global_step:
                max_global_step = global_step
                best_state_file = sf
                best_state = state
        except Exception:
            continue

    if best_state is None:
        raise RuntimeError("Could not read trainer_state.json")

    eval_rows = []

    for item in best_state.get("log_history", []):
        if "eval_loss" in item and "step" in item:
            eval_rows.append({
                "step": int(item["step"]),
                "eval_loss": float(item["eval_loss"]),
                "epoch": item.get("epoch", None),
            })

    if not eval_rows:
        raise RuntimeError("No eval_loss records found in trainer_state.json")

    best_row = min(eval_rows, key=lambda x: x["eval_loss"])

    best_step = best_row["step"]
    best_eval_loss = best_row["eval_loss"]
    best_epoch = best_row["epoch"]

    best_checkpoint_path = output_dir / f"checkpoint-{best_step}"

    if not best_checkpoint_path.exists():
        existing = sorted([p.name for p in output_dir.glob("checkpoint-*")])
        raise FileNotFoundError(
            f"Best checkpoint is missing: {best_checkpoint_path}\n"
            f"Best step from logs = {best_step}, eval_loss = {best_eval_loss}\n"
            f"Existing checkpoints: {existing}"
        )

    return best_checkpoint_path, best_step, best_eval_loss, best_epoch, best_state_file


BEST_CHECKPOINT_PATH, BEST_STEP, BEST_EVAL_LOSS, BEST_EPOCH, BEST_STATE_FILE = find_best_checkpoint_from_logs(OUTPUT_DIR)

print("Best checkpoint:")
print("  path:", BEST_CHECKPOINT_PATH)
print("  step:", BEST_STEP)
print("  epoch:", BEST_EPOCH)
print("  eval_loss:", BEST_EVAL_LOSS)
print("  trainer_state:", BEST_STATE_FILE)

# ------------------------------------------------------------
# Load the best checkpoint into the current trainer/model
# ------------------------------------------------------------

print("\nLoading best checkpoint into model...")

trainer._load_from_checkpoint(str(BEST_CHECKPOINT_PATH), model=trainer.model)
model = trainer.model

print("Best checkpoint loaded successfully.")

# ------------------------------------------------------------
# Save best adapter separately
# ------------------------------------------------------------

BEST_ADAPTER_PATH = ADAPTER_DIR / f"{EXPERIMENT_NAME}_best_step{BEST_STEP}"
BEST_ADAPTER_PATH.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(BEST_ADAPTER_PATH))
tokenizer.save_pretrained(str(BEST_ADAPTER_PATH))

print("\nSaved BEST adapter to:")
print(BEST_ADAPTER_PATH)

Best checkpoint:
  path: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs/checkpoint-600
  step: 600
  epoch: 1.5431145431145432
  eval_loss: 2.019728660583496
  trainer_state: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs/checkpoint-2000/trainer_state.json

Loading best checkpoint into model...
Best checkpoint loaded successfully.


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs_best_step600/tokenizer_config.json.



Saved BEST adapter to:
/content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs_best_step600


### **Quick Inference**

In [19]:
# ============================================================
# Cell 18 — Quick inference function
# Uses the BEST checkpoint loaded in Cell 17
# ============================================================

import torch

if "BEST_CHECKPOINT_PATH" not in globals():
    raise RuntimeError(
        "BEST_CHECKPOINT_PATH is not defined. "
        "Run Cell 17 first to load the best checkpoint before inference."
    )

print("Inference will use BEST checkpoint:")
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("BEST_STEP:", BEST_STEP)
print("BEST_EVAL_LOSS:", BEST_EVAL_LOSS)

try:
    UnslothModel.for_inference(model)
except Exception as e:
    print("for_inference not available or not needed:", repr(e))

def extract_assistant_answer(decoded_text):
    if RESPONSE_MARKER in decoded_text:
        answer = decoded_text.split(RESPONSE_MARKER)[-1]
    else:
        answer = decoded_text

    special_tokens = [
        tokenizer.eos_token,
        tokenizer.pad_token,
        "<|endoftext|>",
        "<|im_end|>",
    ]

    for tok in special_tokens:
        if tok:
            answer = answer.replace(tok, "")

    return answer.strip()

def generate_translation_from_row(row, max_new_tokens=120):
    user_text = make_user_prompt(row)

    prompt = format_sft_text(
        system_text=SYSTEM_PROMPT,
        user_text=user_text,
        assistant_text=None,
        add_eos=False,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    answer = extract_assistant_answer(decoded)

    return answer, decoded

sample = eval_df.sample(1, random_state=SEED).iloc[0].to_dict()

pred, raw = generate_translation_from_row(sample)

print("Country/config:", sample["config"])
print("Dialect:", sample["dialect"])
print("Domain:", sample["domain"])
print("\nEnglish:")
print(sample["source_text"])
print("\nReference Arabic:")
print(sample["target_arabic"])
print("\nPrediction:")
print(pred)

Inference will use BEST checkpoint:
BEST_CHECKPOINT_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs/checkpoint-600
BEST_STEP: 600
BEST_EVAL_LOSS: 2.019728660583496
Country/config: EG
Dialect: Egyptian Arabic (Cairene) Dialect
Domain: Construction and real estate

English:
Engineer, good morning. Before you run your cables on the third floor, let's coordinate the wall chases.

Reference Arabic:
صباح الخير يا هندسه. قبل ما تمد الكابلات في الدور التالت، خلينا نتفق على مجاري الحيطان.

Prediction:
يا باشمهندس الصبح الطيب، قبل ما تمشي الكابلات في الدور التالت، نعمل مع بعض في شغل الشق الجداري.


### Generate predictions for a small eval sample

In [20]:
# ============================================================
# Cell 19 — Generate predictions on FULL eval/test set
# Uses BEST checkpoint loaded in Cell 17
# ============================================================

from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import time

if "BEST_CHECKPOINT_PATH" not in globals():
    raise RuntimeError("Run Cell 17 first. The best checkpoint is not loaded.")

# None = full eval/test set
EVAL_LIMIT = None

SAVE_EVERY = 50
STORE_RAW_OUTPUT = False

eval_tag = f"best_step{BEST_STEP}"

pred_path = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{eval_tag}.csv"

print("Experiment:", EXPERIMENT_NAME)
print("Using BEST checkpoint:", BEST_CHECKPOINT_PATH)
print("Best step:", BEST_STEP)
print("Best eval_loss:", BEST_EVAL_LOSS)
print("Saving predictions to:", pred_path)

# ------------------------------------------------------------
# Prepare full eval/test dataframe
# ------------------------------------------------------------

full_eval_df = eval_df.reset_index(drop=True).copy()

if EVAL_LIMIT is not None:
    full_eval_df = full_eval_df.iloc[:EVAL_LIMIT].copy()

expected_n = len(full_eval_df)

print("Total eval/test examples to evaluate:", expected_n)

# ------------------------------------------------------------
# Resume from existing file if available
# ------------------------------------------------------------

if pred_path.exists():
    existing_df = pd.read_csv(pred_path)

    if "source_id" in existing_df.columns:
        pred_rows = existing_df.to_dict("records")
        done_ids = set(existing_df["source_id"].astype(str).tolist())
        print(f"Resuming from existing file: {len(done_ids)} examples already done.")
    else:
        print("Existing file has no source_id column. Starting from scratch.")
        pred_rows = []
        done_ids = set()
else:
    pred_rows = []
    done_ids = set()
    print("Starting full eval prediction from scratch.")

# ------------------------------------------------------------
# Generate predictions
# ------------------------------------------------------------

start_time = time.time()

for _, row in tqdm(full_eval_df.iterrows(), total=len(full_eval_df)):
    row_dict = row.to_dict()
    source_id = str(row_dict["source_id"])

    if source_id in done_ids:
        continue

    try:
        pred, raw = generate_translation_from_row(row_dict)

        out_row = {
            "source_id": row_dict["source_id"],
            "config": row_dict.get("config", ""),
            "dialect": row_dict.get("dialect", ""),
            "domain": row_dict.get("domain", ""),
            "source_text": row_dict["source_text"],
            "reference_arabic": row_dict["target_arabic"],
            "prediction": pred,
            "model_checkpoint": str(BEST_CHECKPOINT_PATH),
            "best_step": BEST_STEP,
            "best_eval_loss": BEST_EVAL_LOSS,
        }

        if STORE_RAW_OUTPUT:
            out_row["raw_output"] = raw

    except Exception as e:
        out_row = {
            "source_id": row_dict.get("source_id", ""),
            "config": row_dict.get("config", ""),
            "dialect": row_dict.get("dialect", ""),
            "domain": row_dict.get("domain", ""),
            "source_text": row_dict.get("source_text", ""),
            "reference_arabic": row_dict.get("target_arabic", ""),
            "prediction": "",
            "generation_error": repr(e),
            "model_checkpoint": str(BEST_CHECKPOINT_PATH),
            "best_step": BEST_STEP,
            "best_eval_loss": BEST_EVAL_LOSS,
        }

    pred_rows.append(out_row)
    done_ids.add(source_id)

    if len(pred_rows) % SAVE_EVERY == 0:
        tmp_df = pd.DataFrame(pred_rows)
        tmp_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
        print(f"Saved partial predictions: {len(tmp_df)} rows")

# ------------------------------------------------------------
# Final save
# ------------------------------------------------------------

pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

elapsed = time.time() - start_time

# ------------------------------------------------------------
# Full eval safety check
# ------------------------------------------------------------

expected_ids = set(full_eval_df["source_id"].astype(str).tolist())
actual_ids = set(pred_df["source_id"].astype(str).tolist())

missing_ids = expected_ids - actual_ids
extra_ids = actual_ids - expected_ids

print("\nDone.")
print("Saved predictions to:", pred_path)
print("Total rows saved:", len(pred_df))
print("Expected eval/test rows:", expected_n)
print(f"Elapsed time: {elapsed / 60:.2f} minutes")

if missing_ids:
    raise RuntimeError(f"Prediction file is incomplete. Missing {len(missing_ids)} eval examples.")

if extra_ids:
    print(f"Warning: prediction file has {len(extra_ids)} extra source_ids not in current eval_df.")

if len(pred_df) == expected_n:
    print("Full eval/test set was evaluated successfully.")
else:
    print("Warning: row count differs from expected eval size.")

display(pred_df.head())

Experiment: qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs
Using BEST checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs/checkpoint-600
Best step: 600
Best eval_loss: 2.019728660583496
Saving predictions to: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs_best_step600.csv
Total eval/test examples to evaluate: 1118
Starting full eval prediction from scratch.


  0%|          | 0/1118 [00:00<?, ?it/s]

Saved partial predictions: 50 rows
Saved partial predictions: 100 rows
Saved partial predictions: 150 rows
Saved partial predictions: 200 rows
Saved partial predictions: 250 rows
Saved partial predictions: 300 rows
Saved partial predictions: 350 rows
Saved partial predictions: 400 rows
Saved partial predictions: 450 rows
Saved partial predictions: 500 rows
Saved partial predictions: 550 rows
Saved partial predictions: 600 rows
Saved partial predictions: 650 rows
Saved partial predictions: 700 rows
Saved partial predictions: 750 rows
Saved partial predictions: 800 rows
Saved partial predictions: 850 rows
Saved partial predictions: 900 rows
Saved partial predictions: 950 rows
Saved partial predictions: 1000 rows
Saved partial predictions: 1050 rows
Saved partial predictions: 1100 rows

Done.
Saved predictions to: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs_best_step600.csv
Total rows saved:

,source_id,config,dialect,domain,source_text,reference_arabic,prediction,model_checkpoint,best_step,best_eval_loss
0,EG_test_EG_test_0_0,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"I would like one order of kunafa, please.",عايز واحد كنافة لو سمحت.,عايزة طلبة واحدة من الكناfeh لو سمحت.,/content/drive/MyDrive/alexandria_qwen35_sft/r...,600,2.019729
1,EG_test_EG_test_0_1,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,Certainly. Would you like that with cheese or ...,أكيد. تحبها بالجبنة ولا بالقشطة؟,أكيد، تحب بالماشي ولا بالزبادي؟,/content/drive/MyDrive/alexandria_qwen35_sft/r...,600,2.019729
2,EG_test_EG_test_0_2,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"With cream, please.",بالقشطة، لو سمحت.,بصبر، مع الحليب.,/content/drive/MyDrive/alexandria_qwen35_sft/r...,600,2.019729
3,EG_test_EG_test_1_0,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"Pardon me, I believe the meat is overcooked. I...",لو سمحت، أعتقد اللحمة مستوية زيادة. ناشفة جدا.,لو سمحتي، أنا شايف إن اللحم اتحمر شوية. صعب أكله.,/content/drive/MyDrive/alexandria_qwen35_sft/r...,600,2.019729
4,EG_test_EG_test_1_1,EG,Egyptian Arabic (Cairene) Dialect,Commerce and transactions,"I'm very sorry to hear that, sir. Would you li...",آسفه جدا يا فندم. تحب أخلي الشيف يجهزلك واحدة ...,بأسف جدا يا فندم، كنت عايز أطلب من الخادم يجهز...,/content/drive/MyDrive/alexandria_qwen35_sft/r...,600,2.019729


### **Compute BLEU and chrF**

In [21]:
# ============================================================
# Cell 20 — Compute BLEU and chrF on FULL eval/test predictions
# Uses predictions from BEST checkpoint
# ============================================================

import pandas as pd
import json
from pathlib import Path

try:
    import sacrebleu
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"])
    import sacrebleu

if "BEST_STEP" not in globals():
    raise RuntimeError("BEST_STEP is not defined. Run Cell 17 first.")

eval_tag = f"best_step{BEST_STEP}"

pred_path = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{eval_tag}.csv"
metrics_path = PRED_DIR / f"full_eval_metrics_{EXPERIMENT_NAME}_{eval_tag}.json"

print("Experiment:", EXPERIMENT_NAME)
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Best step:", BEST_STEP)
print("Prediction file:", pred_path)
print("Metrics file:", metrics_path)

if not pred_path.exists():
    raise FileNotFoundError(f"Prediction file not found: {pred_path}")

# ------------------------------------------------------------
# Load predictions
# ------------------------------------------------------------

pred_df = pd.read_csv(pred_path)

required_cols = {"source_id", "prediction", "reference_arabic"}
missing = required_cols - set(pred_df.columns)

if missing:
    raise ValueError(f"Missing required columns in prediction file: {missing}")

pred_df["source_id"] = pred_df["source_id"].astype(str)
pred_df["prediction"] = pred_df["prediction"].fillna("").astype(str)
pred_df["reference_arabic"] = pred_df["reference_arabic"].fillna("").astype(str)

# ------------------------------------------------------------
# Full eval/test safety check
# ------------------------------------------------------------

expected_eval_df = eval_df.reset_index(drop=True).copy()
expected_eval_df["source_id"] = expected_eval_df["source_id"].astype(str)

expected_ids = set(expected_eval_df["source_id"].tolist())
actual_ids = set(pred_df["source_id"].tolist())

missing_ids = expected_ids - actual_ids
extra_ids = actual_ids - expected_ids

print("\n==============================")
print("Full Eval/Test Coverage Check")
print("==============================")
print("Expected eval/test examples:", len(expected_eval_df))
print("Prediction rows:", len(pred_df))
print("Unique prediction source_ids:", len(actual_ids))

if missing_ids:
    raise RuntimeError(
        f"Prediction file is NOT full eval/test. "
        f"Missing {len(missing_ids)} examples. "
        f"Run Cell 19 again to finish generation."
    )

if extra_ids:
    print(f"Warning: prediction file has {len(extra_ids)} extra source_ids not in current eval_df.")

print("Full eval/test coverage confirmed.")

# ------------------------------------------------------------
# Reorder predictions to match eval_df order
# ------------------------------------------------------------

order_df = expected_eval_df[["source_id"]].copy()

pred_df_ordered = order_df.merge(pred_df, on="source_id", how="left")

# ------------------------------------------------------------
# Compute corpus-level BLEU and chrF
# ------------------------------------------------------------

preds = pred_df_ordered["prediction"].fillna("").astype(str).tolist()
refs = pred_df_ordered["reference_arabic"].fillna("").astype(str).tolist()

bleu = sacrebleu.corpus_bleu(preds, [refs])
chrf = sacrebleu.corpus_chrf(preds, [refs])

metrics = {
    "experiment": EXPERIMENT_NAME,
    "checkpoint": str(BEST_CHECKPOINT_PATH),
    "best_step": int(BEST_STEP),
    "best_eval_loss": float(BEST_EVAL_LOSS),
    "num_examples": len(pred_df_ordered),
    "BLEU": bleu.score,
    "chrF": chrf.score,
    "prediction_file": str(pred_path),
}

print("\n==============================")
print("Full Eval/Test Metrics from BEST checkpoint")
print("==============================")
print(f"Examples: {len(pred_df_ordered)}")
print(f"BLEU:     {bleu.score:.4f}")
print(f"chrF:     {chrf.score:.4f}")

# ------------------------------------------------------------
# Per-config / per-dialect / per-domain metrics
# ------------------------------------------------------------

def compute_group_metrics(df, group_col):
    rows = []

    if group_col not in df.columns:
        return pd.DataFrame(rows)

    for group_value in sorted(df[group_col].dropna().unique()):
        tmp = df[df[group_col] == group_value]

        if len(tmp) == 0:
            continue

        group_preds = tmp["prediction"].fillna("").astype(str).tolist()
        group_refs = tmp["reference_arabic"].fillna("").astype(str).tolist()

        rows.append({
            group_col: group_value,
            "num_examples": len(tmp),
            "BLEU": sacrebleu.corpus_bleu(group_preds, [group_refs]).score,
            "chrF": sacrebleu.corpus_chrf(group_preds, [group_refs]).score,
        })

    return pd.DataFrame(rows)

per_config_df = compute_group_metrics(pred_df_ordered, "config")
per_dialect_df = compute_group_metrics(pred_df_ordered, "dialect")
per_domain_df = compute_group_metrics(pred_df_ordered, "domain")

if len(per_config_df):
    print("\n==============================")
    print("Per-config Metrics")
    print("==============================")
    display(per_config_df)
    metrics["per_config"] = per_config_df.to_dict("records")

if len(per_dialect_df):
    print("\n==============================")
    print("Per-dialect Metrics")
    print("==============================")
    display(per_dialect_df)
    metrics["per_dialect"] = per_dialect_df.to_dict("records")

if len(per_domain_df):
    print("\n==============================")
    print("Per-domain Metrics")
    print("==============================")
    display(per_domain_df)
    metrics["per_domain"] = per_domain_df.to_dict("records")

# ------------------------------------------------------------
# Save metrics
# ------------------------------------------------------------

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("\nSaved metrics to:")
print(metrics_path)

display(pred_df_ordered[["source_text", "reference_arabic", "prediction"]].head(10))

Experiment: qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs
Best checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs/checkpoint-600
Best step: 600
Prediction file: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs_best_step600.csv
Metrics file: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_metrics_qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs_best_step600.json

Full Eval/Test Coverage Check
Expected eval/test examples: 1118
Prediction rows: 1118
Unique prediction source_ids: 1118
Full eval/test coverage confirmed.

Full Eval/Test Metrics from BEST checkpoint
Examples: 1118
BLEU:     8.2433
chrF:     37.2705

Per-config Metrics


,config,num_examples,BLEU,chrF
0,EG,1118,8.243267,37.270451



Per-dialect Metrics


,dialect,num_examples,BLEU,chrF
0,Egyptian Arabic (Cairene) Dialect,1118,8.243267,37.270451



Per-domain Metrics


,domain,num_examples,BLEU,chrF
0,Agriculture and farming,100,9.016333,36.883442
1,Commerce and transactions,103,8.743318,37.055336
2,Construction and real estate,102,6.464395,36.655565
3,Education and academia,102,9.635916,36.349397
4,Energy and resources,103,8.676291,39.224394
5,Everyday and social,103,6.339766,33.383415
6,Healthcare and medical,102,10.646204,37.769939
7,Legal and financial,103,5.397394,39.386685
8,Logistics and transportation,100,6.392840,36.832000
9,Professional and workplace,100,8.780112,39.225821



Saved metrics to:
/content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_metrics_qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs_best_step600.json


,source_text,reference_arabic,prediction
0,"I would like one order of kunafa, please.",عايز واحد كنافة لو سمحت.,عايزة طلبة واحدة من الكناfeh لو سمحت.
1,Certainly. Would you like that with cheese or ...,أكيد. تحبها بالجبنة ولا بالقشطة؟,أكيد، تحب بالماشي ولا بالزبادي؟
2,"With cream, please.",بالقشطة، لو سمحت.,بصبر، مع الحليب.
3,"Pardon me, I believe the meat is overcooked. I...",لو سمحت، أعتقد اللحمة مستوية زيادة. ناشفة جدا.,لو سمحتي، أنا شايف إن اللحم اتحمر شوية. صعب أكله.
4,"I'm very sorry to hear that, sir. Would you li...",آسفه جدا يا فندم. تحب أخلي الشيف يجهزلك واحدة ...,بأسف جدا يا فندم، كنت عايز أطلب من الخادم يجهز...
5,"Yes, please. Thank you.",أيوه لو سمحتي. شكرا.,اه لو سمحتي، شكرا جدا
6,"Honestly, the driver was rude and the car was ...",بصراحة، السواق كان بجح و العربيه ماكانتش نضيفه...,بصراحة السواق كان لسة صعبه والعربية مش نظيفة، ...
7,I truly apologize that we failed to provide yo...,أنا حقيقي بعتذر ان احنا فشلنا اننا نقدملك خدمة...,انا اسف جدا اننا فشلتنا في خدمةك النهاردة، وبس...
8,"Engineer, good morning. Before you run your ca...",صباح الخير يا هندسه. قبل ما تمد الكابلات في ال...,يا باشمهندس الصبح الطيب، قبل ما تمشي الكابلات ...
9,"Good thing you caught me, I was about to start...",كويس إنك لحقتني، أنا كنت هابدأ. هتحط خطوط المي...,الحلو انك شفته، كنت عندي ابدأ. عندكم فين خطوط ...


### **Error-analysis view by country/domain**

In [22]:
# ============================================================
# Cell 21 — Qualitative error-analysis samples
# Uses FULL eval predictions from BEST checkpoint
# ============================================================

if "BEST_STEP" not in globals():
    raise RuntimeError("BEST_STEP is not defined. Run Cell 17 first.")

eval_tag = f"best_step{BEST_STEP}"

analysis_path = PRED_DIR / f"manual_analysis_{EXPERIMENT_NAME}_{eval_tag}.xlsx"

# Prefer ordered full-eval dataframe from Cell 20
if "pred_df_ordered" in globals():
    analysis_df = pred_df_ordered.copy()
elif "pred_df" in globals():
    analysis_df = pred_df.copy()
else:
    raise RuntimeError("No prediction dataframe found. Run Cell 19/20 first.")

with pd.ExcelWriter(analysis_path, engine="openpyxl") as writer:
    analysis_df.to_excel(writer, sheet_name="all_predictions", index=False)

    if "config" in analysis_df.columns:
        for cfg in analysis_df["config"].dropna().unique()[:10]:
            tmp = analysis_df[analysis_df["config"] == cfg]
            tmp.to_excel(writer, sheet_name=str(cfg)[:31], index=False)

print("Saved manual analysis workbook to:")
print(analysis_path)

Saved manual analysis workbook to:
/content/drive/MyDrive/alexandria_qwen35_sft/predictions/manual_analysis_qwen35_2b_alexandria_eg_only_context3_fnn_group_r16_v2_10epochs_best_step600.xlsx


### **Comparisons**